# Spark Interview Prep — Expert

**Target:** Principal/Staff Engineers and Architects (8+ yrs). Deep internals.

8 topics, ~12 cells each.

In [1]:
import os, time, sys
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, count, sum as _sum, avg, max as _max, min as _min,
    window, lag, lead, rank, dense_rank, row_number,
    broadcast, when, lit, expr, regexp_replace, to_date,
    year, month, dayofmonth, concat_ws, split, trim
)
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = (SparkSession.builder
    .appName('SparkInterviewPrep-Expert')
    .config('spark.sql.adaptive.enabled', 'true')
    .config('spark.sql.shuffle.partitions', '8')
    .config('spark.driver.memory', '2g')
    .config('spark.sql.cbo.enabled', 'true')
    .config('spark.sql.statistics.histogram.enabled', 'true')
    .getOrCreate())

spark.sparkContext.setLogLevel('WARN')

CABS_PATH = r'C:\Users\PS\Documents\Python-Exp\RawData\Cabs.csv'

cabs = (spark.read
    .option('header', 'true')
    .option('inferSchema', 'true')
    .csv(CABS_PATH))

cabs.printSchema()
print(f'Row count: {cabs.count()}')
cabs.show(5, truncate=False)

root
 |-- CabNumber: string (nullable = true)
 |-- VehicleLicenseNumber: string (nullable = true)
 |-- Name: string (nullable = true)
 |-- LicenseType: string (nullable = true)
 |-- Active: string (nullable = true)
 |-- PermitLicenseNumber: string (nullable = true)
 |-- VehicleVinNumber: string (nullable = true)
 |-- WheelchairAccessible: string (nullable = true)
 |-- VehicleYear: integer (nullable = true)
 |-- VehicleType: string (nullable = true)
 |-- TelephoneNumber: string (nullable = true)
 |-- Website: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- LastDateUpdated: string (nullable = true)

Row count: 8970
+---------+--------------------+-------------------+----------------+------+-------------------+-----------------+--------------------+-----------+-----------+---------------+-------+------------------------------------------+---------------+
|CabNumber|VehicleLicenseNumber|Name               |LicenseType     |Active|PermitLicenseNumber|VehicleVinNumber |W

## Topic 1 — Catalyst Optimizer Internals

Catalyst is Spark's extensible query optimizer written in Scala using functional programming and tree transformations.

| Internal Component | Role |
|---|---|
| Unresolved Logical Plan | Raw parsed plan before attribute resolution |
| Analyzed Logical Plan | Attributes resolved against catalog |
| Optimized Logical Plan | Rule-based + cost-based rewrites applied |
| Physical Plan(s) | One or more execution strategies selected |
| Selected Physical Plan | Best plan after cost comparison |
| Generated Code | Whole-stage code gen via Janino compiler |

### Business Scenario

Your team is debugging a slow query on the Cabs dataset. A junior engineer says 'the filter runs after the join, that is why it is slow.'
You need to prove or disprove this by reading the Catalyst output — and explain what predicate pushdown does.

**Architecture Challenge:** How do you inspect each phase of Catalyst to verify a rule fired correctly?

In [2]:
# Catalyst 4-phase demo: Analysis -> Logical Opt -> Physical Planning -> Code Gen
query = (cabs
    .filter(col('Active') == 'YES')
    .select('Name', 'LicenseType')
    .filter(col('LicenseType') == 'LIMOUSINE'))

print('=== EXTENDED EXPLAIN (all 4 phases) ===')
query.explain(extended=True)

print('\n=== CODEGEN EXPLAIN ===')
query.explain('codegen')

=== EXTENDED EXPLAIN (all 4 phases) ===
== Parsed Logical Plan ==
'Filter '`=`('LicenseType, LIMOUSINE)
+- Project [Name#19, LicenseType#20]
   +- Filter (Active#21 = YES)
      +- Relation [CabNumber#17,VehicleLicenseNumber#18,Name#19,LicenseType#20,Active#21,PermitLicenseNumber#22,VehicleVinNumber#23,WheelchairAccessible#24,VehicleYear#25,VehicleType#26,TelephoneNumber#27,Website#28,Address#29,LastDateUpdated#30] csv

== Analyzed Logical Plan ==
Name: string, LicenseType: string
Filter (LicenseType#20 = LIMOUSINE)
+- Project [Name#19, LicenseType#20]
   +- Filter (Active#21 = YES)
      +- Relation [CabNumber#17,VehicleLicenseNumber#18,Name#19,LicenseType#20,Active#21,PermitLicenseNumber#22,VehicleVinNumber#23,WheelchairAccessible#24,VehicleYear#25,VehicleType#26,TelephoneNumber#27,Website#28,Address#29,LastDateUpdated#30] csv

== Optimized Logical Plan ==
Project [Name#19, LicenseType#20]
+- Filter ((isnotnull(Active#21) AND isnotnull(LicenseType#20)) AND ((Active#21 = YES) AND (Lic

### Expert Interview Questions — Catalyst Internals

1. Walk me through all 4 phases of Catalyst query compilation. What tree transformations happen in each phase?
2. What is the difference between rule-based optimization (RBO) and cost-based optimization (CBO) in Catalyst?
3. How does predicate pushdown reordering work internally? Which Catalyst rule handles it and what tree pattern does it match?
4. Explain constant folding. Give an example of a query expression that Catalyst simplifies at compile time.
5. How would you add a custom Catalyst optimization rule in a Spark extension? What trait must you implement?
6. What is the role of the Analyzer phase vs the Optimizer phase? Can the Analyzer apply optimizations?
7. Why does Catalyst use immutable tree nodes with pattern matching instead of mutable AST nodes?
8. How does the physical planner select between multiple physical plan candidates? What is the role of SparkStrategies?

### Catalyst Internals Deep Dive

**Phase 1 — Analysis:** Resolves `UnresolvedAttribute` and `UnresolvedRelation` nodes using the Catalog.
Rules include `ResolveRelations`, `ResolveReferences`, `ResolveAliases`.

**Phase 2 — Logical Optimization:** Applies a fixed-point batch of rules until convergence or max iterations.
Key rules: `PushDownPredicate`, `ColumnPruning`, `ConstantFolding`, `BooleanSimplification`, `CombineFilters`.

**Phase 3 — Physical Planning:** `SparkPlanner` applies `Strategy` objects to convert logical to physical operators.
Strategies: `JoinSelection`, `BasicOperators`, `Aggregation`, `FileSourceScanExec`.

**Phase 4 — Code Generation:** `CollapseCodegenStages` wraps eligible operators in `WholeStageCodegenExec`.
Janino compiles the generated Java bytecode at runtime for cache-friendly tight loops.

### Physical Plan Key Operators

When reading `explain()` output, look for these physical operators:

- `*(N) Project` — WholeStageCodegen stage N, projection
- `*(N) Filter` — codegen filter, often fused with scan
- `*(N) FileScan csv` — file scan with pushed-down filters shown as `PushedFilters`
- `Exchange` — shuffle boundary (no codegen wrapping)
- `BroadcastExchange` — broadcast of a small relation
- `SortMergeJoin` — sort-merge join requiring sorted inputs

The `*(N)` prefix confirms WholeStageCodegen is active for that operator cluster.

### Spark UI — What to Look For (Catalyst)

- **SQL tab > query details:** View the physical plan DAG. Confirm filters appear before joins/scans.
- **Pushed Filters:** In `FileScan` node — if filters are NOT pushed, check column types (some types block pushdown).
- **Duration of stages:** A plan with predicate pushdown will show fewer rows entering downstream stages.
- **Metrics on scan node:** `number of output rows` should be much less than `number of files read` rows if pushdown works.
- **WholeStageCodegen boundary:** Look for Exchange nodes that break codegen — minimize these.

In [3]:
def audit_catalyst_plan(df, label='DataFrame'):
    print(f'=== Catalyst Plan Audit: {label} ===')
    print('--- Simple (physical only) ---')
    df.explain()
    print('--- Extended (all phases) ---')
    df.explain(extended=True)
    print('--- Cost (with stats if CBO enabled) ---')
    df.explain('cost')
    print('--- Formatted (tree-style, Spark 3.x) ---')
    df.explain('formatted')

# Demo on a multi-step query
demo_q = (cabs
    .filter(col('Active') == 'YES')
    .filter(col('VehicleYear') > 2015)
    .select('Name', 'LicenseType', 'VehicleYear'))

audit_catalyst_plan(demo_q, 'active-cabs-post-2015')

=== Catalyst Plan Audit: active-cabs-post-2015 ===
--- Simple (physical only) ---
== Physical Plan ==
*(1) Project [Name#19, LicenseType#20, VehicleYear#25]
+- *(1) Filter (((isnotnull(Active#21) AND isnotnull(VehicleYear#25)) AND (Active#21 = YES)) AND (VehicleYear#25 > 2015))
   +- FileScan csv [Name#19,LicenseType#20,Active#21,VehicleYear#25] Batched: false, DataFilters: [isnotnull(Active#21), isnotnull(VehicleYear#25), (Active#21 = YES), (VehicleYear#25 > 2015)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/PS/Documents/Python-Exp/RawData/Cabs.csv], PartitionFilters: [], PushedFilters: [IsNotNull(Active), IsNotNull(VehicleYear), EqualTo(Active,YES), GreaterThan(VehicleYear,2015)], ReadSchema: struct<Name:string,LicenseType:string,Active:string,VehicleYear:int>


--- Extended (all phases) ---
== Parsed Logical Plan ==
'Project ['Name, 'LicenseType, 'VehicleYear]
+- Filter (VehicleYear#25 > 2015)
   +- Filter (Active#21 = YES)
      +- Relation [CabNumber#17,Vehic

### Deep Dive — Reading the Extended Plan

In `extended=True` output, 3 logical plans appear before the physical plan:

1. **Parsed Logical Plan** — direct AST from the DSL call chain, attributes unresolved.
2. **Analyzed Logical Plan** — all attributes resolved; types are known.
3. **Optimized Logical Plan** — rewritten by optimizer rules. Here you see `CombineFilters` merging two filters into one, and `PushedFilters` annotating the scan.
4. **Physical Plan** — executor-level operators with Exchange and codegen boundaries.

When two `.filter()` calls collapse into one in the optimized plan, `CombineFilters` fired.
When the filter moves inside the `FileScan`, `PushDownPredicate` fired.

### Expert Best Practices — Catalyst

- Always run `explain('formatted')` (Spark 3+) on production queries; it is far more readable than plain `explain()`.
- Never apply Python-level pre-filtering as a substitute for Spark predicate pushdown — trust the optimizer.
- If a filter is NOT pushed into a Parquet scan, check: is the column a complex type (array/map/struct)? Spark does not push those.
- Use `spark.sql.optimizer.maxIterations` (default 100) cautiously — increasing it for complex queries has diminishing returns.
- CBO stats must be CURRENT. Stale statistics after large data loads will mislead the optimizer — re-run ANALYZE.
- Custom Catalyst rules must be idempotent — Catalyst applies them in fixed-point loops and a non-idempotent rule can loop infinitely.
- Anti-pattern: using `df.rdd.map(...)` to avoid Catalyst — this bypasses all optimizer phases and kills performance.
- Anti-pattern: chaining `.limit()` before a join in an attempt to 'help' the optimizer — it actually blocks predicate pushdown.

In [4]:
# Production-grade Catalyst inspection utility
def catalyst_report(df, label='query'):
    import re
    from io import StringIO
    import sys
    old_stdout = sys.stdout
    sys.stdout = buf = StringIO()
    df.explain(extended=True)
    sys.stdout = old_stdout
    plan_text = buf.getvalue()

    pushed = re.findall(r'PushedFilters: \[([^\]]+)\]', plan_text)
    codegen_stages = re.findall(r'\*(\d+)', plan_text)
    exchanges = plan_text.count('Exchange')
    broadcasts = plan_text.count('BroadcastExchange')

    print(f'=== Catalyst Report: {label} ===')
    print(f'  Pushed filters   : {pushed if pushed else "NONE — check column types"}')
    print(f'  Codegen stages   : {sorted(set(codegen_stages))}')
    print(f'  Exchange nodes   : {exchanges} (each = shuffle boundary)')
    print(f'  BroadcastExchange: {broadcasts}')
    if exchanges > 4:
        print('  WARNING: High shuffle count — review join and aggregation strategy')
    return plan_text

report = catalyst_report(demo_q, 'active-cabs-post-2015')

=== Catalyst Report: active-cabs-post-2015 ===
  Pushed filters   : ['IsNotNull(Active), IsNotNull(VehicleYear), EqualTo(Active,YES), GreaterThan(VehicleYear,2015)']
  Codegen stages   : []
  Exchange nodes   : 0 (each = shuffle boundary)
  BroadcastExchange: 0


### Practice Exercises — Catalyst

1. Write a query that joins `cabs` to a synthetic lookup DataFrame. Run `explain(extended=True)` and locate the join strategy chosen. Then add a `broadcast()` hint and observe which Catalyst rule changed in the physical plan.
2. Create a query with three consecutive `.filter()` calls. Inspect the optimized logical plan. How many filter nodes remain? Which rule collapsed them? Now deliberately add a `.cache()` between two filters and observe how it prevents further rule application.
3. Use `spark.range(10_000_000).selectExpr('id % 100 as key', 'id * 2 as val')`. Write an aggregation. Use `catalyst_report()` to count Exchange nodes. Then set `spark.sql.shuffle.partitions` to 200 and 2 and compare codegen stage counts and Exchange counts.

## Topic 2 — Tungsten Engine

Project Tungsten (Spark 1.5+) rewrites Spark's execution layer for CPU and memory efficiency.

| Internal Component | Role |
|---|---|
| UnsafeRow | Binary row format stored off-heap, no JVM object overhead |
| WholeStageCodegen | Fuses multiple operators into one compiled JVM method |
| Cache-conscious sort | TimSort on binary data without deserialization |
| BytesToBytesMap | Off-heap hash map for aggregations |
| MemoryManager | Tracks on-heap + off-heap allocations per task |

### Business Scenario

A data platform team reports that after upgrading from Spark 2.x to 3.x a complex aggregation pipeline is 3x faster.
The manager asks why. The engineer needs to explain Project Tungsten's role and quantify which specific feature drove the speedup.

**Architecture Challenge:** How do you verify that WholeStageCodegen is active for a given query and what do you do when it is disabled for a stage?

In [5]:
# Demonstrate WholeStageCodegen presence/absence
large_df = spark.range(1_000_000).selectExpr(
    'id',
    'id % 50 as group_key',
    'cast(id * 1.5 as double) as value'
)

print('=== WITH WholeStageCodegen (default) ===')
spark.conf.set('spark.sql.codegen.wholeStage', 'true')
wscg_on = large_df.groupBy('group_key').agg(_sum('value').alias('total'))
wscg_on.explain()

print('\n=== WITHOUT WholeStageCodegen ===')
spark.conf.set('spark.sql.codegen.wholeStage', 'false')
wscg_off = large_df.groupBy('group_key').agg(_sum('value').alias('total'))
wscg_off.explain()

# Restore
spark.conf.set('spark.sql.codegen.wholeStage', 'true')

=== WITH WholeStageCodegen (default) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[group_key#108L], functions=[sum(value#109)])
   +- Exchange hashpartitioning(group_key#108L, 8), ENSURE_REQUIREMENTS, [plan_id=99]
      +- HashAggregate(keys=[group_key#108L], functions=[partial_sum(value#109)])
         +- Project [(id#107L % 50) AS group_key#108L, cast((cast(id#107L as decimal(20,0)) * 1.5) as double) AS value#109]
            +- Range (0, 1000000, step=1, splits=8)



=== WITHOUT WholeStageCodegen ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[group_key#108L], functions=[sum(value#109)])
   +- Exchange hashpartitioning(group_key#108L, 8), ENSURE_REQUIREMENTS, [plan_id=116]
      +- HashAggregate(keys=[group_key#108L], functions=[partial_sum(value#109)])
         +- Project [(id#107L % 50) AS group_key#108L, cast((cast(id#107L as decimal(20,0)) * 1.5) as double) AS value#109]
            +- Range (0, 1000000, ste

### Expert Interview Questions — Tungsten

1. What is `UnsafeRow`? How does it differ from a standard JVM object row and why does that matter for GC pressure?
2. Explain whole-stage code generation. What does the generated Java code look like for a simple filter + project pipeline?
3. When is WholeStageCodegen explicitly disabled? Name at least three conditions that force Spark to fall back to interpreted mode.
4. What are the goals of Project Tungsten beyond just off-heap memory? How does it improve CPU cache utilization?
5. What is `BytesToBytesMap` and how does it improve hash aggregation performance over a standard Java HashMap?
6. How does cache-conscious sort work on binary data? Why is avoiding deserialization beneficial for sorting?
7. What is the `spark.sql.codegen.maxFields` setting and what problem does it solve?
8. How does Tungsten interact with Catalyst? At what stage does code generation occur in the query compilation pipeline?

### Tungsten Internals Deep Dive

**UnsafeRow format:**
- Fixed-length null bitset (1 bit per field)
- Fixed-length field values (8 bytes each, or offset+length for variable-length)
- Variable-length data appended at end
- No Java object header (16 bytes saved per row), no GC scanning

**WholeStageCodegen mechanism:**
- `CollapseCodegenStages` rule identifies operator chains with no Exchange boundaries
- Each chain gets a `WholeStageCodegenExec` wrapper
- `CodegenSupport` trait defines `produce()` / `consume()` protocol
- Operators call each other's `produce()`/`consume()` to build a single tight loop
- Janino compiles the resulting Java source to bytecode at runtime

**When codegen is disabled:**
- Too many fields (`spark.sql.codegen.maxFields`, default 100)
- Unsupported expression types (some UDFs, complex types)
- `Exchange` / `Sort` that breaks the pipeline

### Physical Plan — Tungsten Markers

The `*(N)` prefix in explain output marks WholeStageCodegen boundaries:

```
*(2) HashAggregate(keys=[group_key], functions=[sum(value)])
+- Exchange hashpartitioning(group_key, 8)
   +- *(1) HashAggregate(keys=[group_key], functions=[partial_sum(value)])
      +- *(1) Project [id%50 AS group_key, cast(id*1.5 AS double) AS value]
         +- *(1) Range (0, 1000000, step=1, splits=4)
```

Stage 1 fuses: Range + Project + partial HashAggregate into one compiled method.
Exchange breaks the chain (shuffle boundary).
Stage 2 fuses: final HashAggregate.

### Spark UI — Tungsten Metrics

- **SQL tab > metrics per node:** Look for `time in WholeStageCodegen` metric on supported operators.
- **Executor tab > GC time:** Low GC% (under 5%) confirms off-heap UnsafeRow is reducing JVM object pressure.
- **Stage detail > task metrics:** `shuffle spill (memory)` and `shuffle spill (disk)` indicate memory pressure breaking Tungsten's in-memory sort.
- **Peak execution memory per task:** Elevated values suggest BytesToBytesMap is growing large — consider increasing `spark.sql.execution.arrow.maxRecordsPerBatch` or adding salt.
- If you see tasks with codegen disabled (no `*(N)` prefix in plan), check field count and expression types.

In [6]:
def tungsten_audit(df):
    from io import StringIO
    import sys, re
    buf = StringIO()
    old = sys.stdout; sys.stdout = buf
    df.explain()
    sys.stdout = old
    plan = buf.getvalue()

    wscg_stages = re.findall(r'\*(\d+)', plan)
    exchanges = plan.count('Exchange')
    has_sort = 'Sort' in plan
    has_bnlj = 'BroadcastNestedLoop' in plan

    print('=== Tungsten Audit ===')
    if wscg_stages:
        print(f'  WholeStageCodegen ACTIVE  — stages: {sorted(set(wscg_stages))}')
    else:
        print('  WholeStageCodegen INACTIVE — check field count and expression types')
    print(f'  Exchange boundaries: {exchanges}')
    print(f'  Sort present      : {has_sort}')
    print(f'  BNLJ present      : {has_bnlj} (BNLJ disables codegen)')

tungsten_audit(large_df.groupBy('group_key').agg(_sum('value')))

# Timing comparison: codegen on vs off
spark.conf.set('spark.sql.codegen.wholeStage', 'true')
t0 = time.time()
large_df.groupBy('group_key').agg(_sum('value')).collect()
t_on = time.time() - t0

spark.conf.set('spark.sql.codegen.wholeStage', 'false')
t0 = time.time()
large_df.groupBy('group_key').agg(_sum('value')).collect()
t_off = time.time() - t0

spark.conf.set('spark.sql.codegen.wholeStage', 'true')
print(f'Codegen ON : {t_on:.2f}s')
print(f'Codegen OFF: {t_off:.2f}s')
print(f'Speedup    : {t_off/t_on:.2f}x')

=== Tungsten Audit ===
  WholeStageCodegen INACTIVE — check field count and expression types
  Exchange boundaries: 1
  Sort present      : False
  BNLJ present      : False (BNLJ disables codegen)
Codegen ON : 0.57s
Codegen OFF: 0.46s
Speedup    : 0.81x


### Results Deep Dive — Tungsten

The timing comparison shows the impact of WholeStageCodegen on a groupBy + sum workload.

With codegen ON, the generated method looks roughly like:
```java
void processNext() {
  while (inputIterator.hasNext()) {
    InternalRow row = inputIterator.next();
    long id = row.getLong(0);
    long group_key = id % 50;
    double value = (double)(id) * 1.5;
    map.merge(group_key, value);  // direct hash map insert
  }
}
```
No virtual dispatch, no boxing, no iterator overhead between operators.
The CPU can keep data in L1/L2 cache across the entire pipeline for each input batch.

### Expert Best Practices — Tungsten

- Keep column count under `spark.sql.codegen.maxFields` (default 100) to preserve WholeStageCodegen.
- Avoid Python/Scala UDFs where possible — they break the codegen pipeline and force row serialization.
- Use Pandas UDFs (Arrow-based) when UDFs are necessary; they preserve columnar processing.
- Monitor GC time in the Executors tab; GC > 10% of task time indicates too many JVM objects — consider enabling off-heap with `spark.memory.offHeap.enabled=true`.
- Anti-pattern: calling `.toPandas()` on large DataFrames — it defeats UnsafeRow and causes massive JVM heap allocation.
- Anti-pattern: using RDD API for transformations that have DataFrame equivalents — bypasses all Tungsten optimizations.
- For Spark Streaming, Tungsten is active in Structured Streaming but NOT in legacy DStream API.
- Set `spark.sql.codegen.hugeMethodLimit` (default 65536 bytecode bytes) cautiously — increasing it can cause JIT deoptimization.

In [7]:
# Production: verify Tungsten health for a pipeline
def tungsten_health_check(spark):
    configs = {
        'spark.sql.codegen.wholeStage': 'true',
        'spark.sql.codegen.maxFields': '100',
        'spark.sql.execution.arrow.pyspark.enabled': 'false',
    }
    print('=== Tungsten Health Check ===')
    all_ok = True
    for k, expected in configs.items():
        actual = spark.conf.get(k, 'NOT SET')
        status = 'OK' if actual == expected else f'WARN (got {actual}, expected {expected})'
        if 'WARN' in status: all_ok = False
        print(f'  {k} = {actual}  [{status}]')
    print(f'Overall: {"HEALTHY" if all_ok else "NEEDS REVIEW"}')

tungsten_health_check(spark)

=== Tungsten Health Check ===
  spark.sql.codegen.wholeStage = true  [OK]


IllegalArgumentException: [INVALID_CONF_VALUE.TYPE_MISMATCH] The value 'NOT SET' in the config "spark.sql.codegen.maxFields" is invalid. It should be a/an 'int' value. SQLSTATE: 22022

### Practice Exercises — Tungsten

1. Take `spark.range(5_000_000)` and add 110 columns using `selectExpr`. Run `explain()` and observe that `*(N)` prefixes disappear. Now reduce to 90 columns and confirm WholeStageCodegen returns. Measure the runtime difference.
2. Write a Python UDF that doubles a value. Apply it to `large_df`. Compare `explain()` output with the equivalent `col('value') * 2`. Use `tungsten_audit()` to quantify the codegen loss.
3. Enable `spark.memory.offHeap.enabled=true` and set `spark.memory.offHeap.size=512m`. Rerun the groupBy timing test and compare. Use the Executors UI tab to measure GC% change.

## Topic 3 — Cost-Based Optimizer (CBO)

CBO uses collected table statistics (row count, column-level histograms) to choose optimal join order and strategy.

| Internal Component | Role |
|---|---|
| AnalyzeTable / AnalyzeColumn | Collects row count + column stats into catalog |
| ColumnStat | Holds min, max, ndv, nullCount, avgLen, maxLen, histogram |
| Histogram (equi-height) | Bucket-based distribution for selectivity estimation |
| CostBasedJoinReorder | Reorders multi-way joins by estimated output size |
| JoinSelection (physical) | Uses stats to choose BHJ vs SMJ vs SHJ |

### Business Scenario

A 5-table star-schema join is running in 45 minutes. The DBA says 'Spark is joining in the wrong order — it starts with the two largest tables.'
The fix requires CBO to reorder joins. But CBO is only as good as the statistics it has.

**Architecture Challenge:** How do you collect, verify, and refresh statistics in a Spark Hive metastore environment? What happens when statistics are stale?

In [ ]:
# CBO demo: register cabs as a temp view and analyze it
cabs.createOrReplaceTempView('cabs_view')

# Check current CBO settings
print('CBO enabled:', spark.conf.get('spark.sql.cbo.enabled'))
print('Histogram enabled:', spark.conf.get('spark.sql.statistics.histogram.enabled'))

# For Hive-managed tables you would run:
# spark.sql('ANALYZE TABLE cabs_managed COMPUTE STATISTICS')
# spark.sql('ANALYZE TABLE cabs_managed COMPUTE STATISTICS FOR COLUMNS CabNumber, LicenseType, Active, VehicleYear')

# For temp views we can still observe cost estimates via explain('cost')
query_cbo = (cabs
    .filter(col('Active') == 'YES')
    .filter(col('VehicleYear') > 2010)
    .select('CabNumber', 'Name', 'LicenseType', 'VehicleYear'))

print('\n=== COST EXPLAIN (shows Statistics in logical plan if CBO active) ===')
query_cbo.explain('cost')

### Expert Interview Questions — CBO

1. What data does CBO collect when you run `ANALYZE TABLE ... COMPUTE STATISTICS FOR COLUMNS`? List all statistics gathered per column.
2. How does Spark use equi-height histograms to estimate join selectivity? What is the difference between equi-height and equi-width histograms?
3. How does AQE (Adaptive Query Execution) differ from CBO? Can they work together? What does AQE do that CBO cannot?
4. Explain CBO join reordering. What algorithm does Spark use to enumerate join orders (hint: it is a dynamic programming approach)?
5. When is CBO counterproductive? Describe a scenario where collected statistics mislead the optimizer into a worse plan.
6. What is `spark.sql.cbo.joinReorder.enabled` and how does it interact with `spark.sql.cbo.enabled`?
7. How do you verify that CBO statistics are being used in the query plan? What text in `explain('cost')` output confirms this?
8. If statistics are collected on a Parquet table and then 50% of the data is appended, how outdated are the statistics and what is the risk?

### CBO Internals Deep Dive

**Statistics storage:** Stored in Hive Metastore as table properties (`spark.sql.statistics.numRows`, etc.) and column properties.

**ColumnStat fields:** `distinctCount`, `min`, `max`, `nullCount`, `avgLen`, `maxLen`, `histogram`

**Selectivity estimation with histograms:**
- Equi-height: each bucket holds equal row count; bucket boundaries vary
- Filter `col > X` estimated by finding which bucket X falls in, summing remaining buckets
- Join cardinality: estimated as `|left| * |right| / max(ndv_left, ndv_right)`

**Join reordering (DP algorithm):**
- `ReorderJoin` logical rule + `CostBasedJoinReorder` optimizer rule
- Enumerate all `2^n` subsets of n relations, pick minimum cost ordering
- Cost metric: estimated output row count * estimated row size = bytes

**AQE vs CBO:**
- CBO works at compile time using static statistics
- AQE works at runtime using actual shuffle map output sizes
- AQE can dynamically switch join strategies mid-execution; CBO cannot

### Physical Plan — CBO Evidence

In `explain('cost')` output, look for `Statistics(sizeInBytes=...)` annotations on logical plan nodes.

Without stats: `Statistics(sizeInBytes=8.0 EiB)` — Spark uses the default maximum (unknown).
With stats: `Statistics(sizeInBytes=45.2 KiB, rowCount=1234)` — actual collected values.

When `rowCount` is present, CBO is actively using statistics for that relation.
When you see the default `8.0 EiB`, CBO is flying blind for that table.

### Spark UI — CBO Signals

- **SQL tab > query plan:** Hover over scan nodes — if `sizeInBytes` shows realistic values, stats are loaded.
- **Join strategy selection:** If CBO is working correctly, smaller tables should appear on the build side of BroadcastHashJoin.
- **AQE in UI:** Look for `AdaptiveSparkPlan` wrapper node. Click through to see the runtime-adjusted plan.
- **Stage skipping:** AQE + CBO together may skip stages entirely if runtime stats show a partition is empty.
- Monitor `spark.sql.statistics.fallBackToHdfs` — if true, Spark falls back to HDFS file size when catalog stats are missing.

In [ ]:
def cbo_health_check(spark, view_or_table_name):
    from io import StringIO
    import sys, re

    test_df = spark.sql(f'SELECT * FROM {view_or_table_name} LIMIT 1')
    buf = StringIO()
    old = sys.stdout; sys.stdout = buf
    test_df.explain('cost')
    sys.stdout = old
    plan = buf.getvalue()

    has_row_count = 'rowCount' in plan
    has_default_size = '8.0 EiB' in plan
    size_matches = re.findall(r'sizeInBytes=([^,)]+)', plan)

    print(f'=== CBO Health Check: {view_or_table_name} ===')
    print(f'  rowCount in plan   : {has_row_count}')
    print(f'  Default size (8 EiB): {has_default_size} — if True, stats missing')
    print(f'  Size estimates     : {size_matches}')
    cbo_on = spark.conf.get('spark.sql.cbo.enabled', 'false')
    print(f'  spark.sql.cbo.enabled: {cbo_on}')
    if has_default_size and cbo_on == 'true':
        print('  ACTION: Run ANALYZE TABLE ... COMPUTE STATISTICS FOR COLUMNS')

cbo_health_check(spark, 'cabs_view')

### Results Deep Dive — CBO

The `cbo_health_check` output tells us whether Spark has real statistics or is using default unknowns.

For temp views (`createOrReplaceTempView`) without explicit stats collection, Spark estimates size from the physical scan (file size / compression ratio estimate). This is better than nothing but far less accurate than column-level histograms.

For production Hive/Delta tables, the workflow is:
1. `ANALYZE TABLE t COMPUTE STATISTICS` — row count and table size
2. `ANALYZE TABLE t COMPUTE STATISTICS FOR COLUMNS col1, col2, col3` — per-column NDV, histograms
3. Verify with `DESCRIBE EXTENDED t` or `explain('cost')`
4. Schedule re-analysis after major data loads (e.g., after daily ETL partition append)

### Expert Best Practices — CBO

- Always run `ANALYZE TABLE ... COMPUTE STATISTICS FOR COLUMNS` on fact tables in star schemas — CBO join reordering only kicks in with column stats.
- Schedule stats refresh in your ETL pipeline immediately after table writes, not on a fixed calendar schedule.
- Use AQE (`spark.sql.adaptive.enabled=true`) as a runtime safety net — it corrects CBO mistakes using actual shuffle sizes.
- Do not enable `spark.sql.cbo.joinReorder.enabled` for joins with more than 8 tables without profiling — the DP algorithm is `O(2^n)` and can add significant planning overhead.
- Anti-pattern: relying on CBO alone for skewed data — histograms may not capture extreme outliers; use AQE skew join handling instead.
- Anti-pattern: collecting stats on staging tables that are frequently overwritten without refreshing stats post-overwrite.
- For Delta Lake, use `ANALYZE TABLE ... COMPUTE STATISTICS` after `OPTIMIZE` + `ZORDER BY` operations.
- Verify stat freshness: if `spark.sql.statistics.fallBackToHdfs=true`, Spark silently uses HDFS file size instead of catalog stats.

In [ ]:
# Production CBO pipeline: analyze + verify + query with cost explain
def run_cbo_optimized_query(spark, view_name, filter_col, filter_val):
    # For real Hive tables, would run ANALYZE first
    # spark.sql(f'ANALYZE TABLE {view_name} COMPUTE STATISTICS')
    # spark.sql(f'ANALYZE TABLE {view_name} COMPUTE STATISTICS FOR COLUMNS ...')

    df = spark.sql(f'SELECT * FROM {view_name} WHERE {filter_col} = "{filter_val}"')

    from io import StringIO
    import sys
    buf = StringIO()
    old = sys.stdout; sys.stdout = buf
    df.explain('cost')
    sys.stdout = old
    plan = buf.getvalue()

    print(f'Query: SELECT * FROM {view_name} WHERE {filter_col} = {filter_val}')
    print(f'Row count result: {df.count()}')
    print('Cost plan snippet (first 800 chars):')
    print(plan[:800])
    return df

result = run_cbo_optimized_query(spark, 'cabs_view', 'Active', 'YES')

### Practice Exercises — CBO

1. Create two synthetic DataFrames: `left = spark.range(1_000_000)` and `right = spark.range(100)`. Register both as temp views. Run `explain('cost')` on a join. Then swap which side is broadcast and observe how the cost estimate changes.
2. Enable CBO and join reordering (`spark.sql.cbo.joinReorder.enabled=true`). Create a 3-table synthetic join where table sizes are 1M, 10K, and 500 rows. Use `explain('cost')` to verify that Spark reorders the join to start with the smallest tables.
3. Simulate stale statistics: collect stats on a 1000-row DataFrame, then double its size by union with itself, but do NOT re-analyze. Run `explain('cost')` and observe the incorrect row count estimate. Document what query plan decision this causes.

## Topic 4 — Join Strategy Selection Deep Dive

Spark selects from 4 join strategies based on data size, join type, and hints.

| Strategy | Trigger | Shuffle? | Sort? | Best For |
|---|---|---|---|---|
| BroadcastHashJoin (BHJ) | Small table < autoBroadcastJoinThreshold | No | No | Small dimension tables |
| SortMergeJoin (SMJ) | Both sides large, equi-join | Yes (both) | Yes (both) | Large-large equi joins |
| ShuffledHashJoin (SHJ) | One side fits in memory after shuffle | Yes (both) | No | Medium tables, no sort needed |
| BroadcastNestedLoopJoin (BNLJ) | Non-equi join or cross join | Broadcast one side | No | Theta joins, cross joins |

### Business Scenario

A pipeline joins a 500GB fact table to a 200MB dimension table. It is using SortMergeJoin and taking 2 hours.
The threshold for auto-broadcast is 10MB (default). The engineer needs to force BroadcastHashJoin.

**Architecture Challenge:** What are the risks of forcing broadcast on a 200MB table? When does broadcast cause OOM and how do you size the broadcast limit safely?

In [ ]:
from pyspark.sql.functions import broadcast

# Create two DataFrames to demonstrate each join strategy
# Left: larger cab facts
left_df = cabs.withColumnRenamed('CabNumber', 'cab_id')

# Right: small lookup of license types
license_data = [('TAXI', 'Taxi Service', 1), ('LIMOUSINE', 'Limo Service', 2), ('BUS', 'Bus Service', 3)]
license_df = spark.createDataFrame(license_data, ['LicenseType', 'ServiceDesc', 'ServiceCode'])

print('=== Strategy 1: BroadcastHashJoin (hint) ===')
bhj = left_df.join(broadcast(license_df), 'LicenseType', 'left')
bhj.explain()

print('\n=== Strategy 2: SortMergeJoin (hint) ===')
from pyspark.sql.functions import col as _col
smj = left_df.hint('merge').join(license_df.hint('merge'), 'LicenseType', 'left')
smj.explain()

print('\n=== Strategy 3: ShuffledHashJoin (hint) ===')
shj = left_df.hint('shuffle_hash').join(license_df.hint('shuffle_hash'), 'LicenseType', 'left')
shj.explain()

print('\n=== Strategy 4: BroadcastNestedLoopJoin (cross join = BNLJ) ===')
bnlj = left_df.crossJoin(broadcast(license_df))
bnlj.explain()

### Expert Interview Questions — Join Strategies

1. Spark has 4 join strategies. Walk through exactly when Spark's `JoinSelection` strategy chooses each one. What is the decision order?
2. What is `spark.sql.autoBroadcastJoinThreshold` and how does Spark measure table size for this comparison? What happens with Delta tables that have no cached size?
3. When does Spark choose ShuffledHashJoin over SortMergeJoin? Is SHJ always faster than SMJ?
4. Explain BroadcastNestedLoopJoin. What is its time complexity? When is it the only option?
5. What are the risks of using the `broadcast()` hint on a table that is actually large? What error do you see and how do you tune the driver to handle it?
6. How does AQE dynamically switch from SortMergeJoin to BroadcastHashJoin at runtime? What is the `spark.sql.adaptive.autoBroadcastJoinThreshold` config?
7. Can you force SortMergeJoin on a small table that Spark wants to broadcast? Why would you ever want to do this?
8. What is a skewed join and how does AQE's skew join optimization split skewed partitions? What config controls it?

### Join Strategy Internals

**JoinSelection decision order (physical planning):**
1. If either side can be broadcast (size < threshold or hint) AND equi-join: BroadcastHashJoin
2. If `spark.sql.join.preferSortMergeJoin=false` AND one side fits in memory AND equi-join: ShuffledHashJoin
3. If both sides can be sorted AND equi-join: SortMergeJoin
4. Otherwise: BroadcastNestedLoopJoin (falls back to this for non-equi / cross joins)

**BHJ execution:**
- Driver broadcasts small side to all executors as a byte array
- Each executor builds a local hash table from broadcast data
- Probe phase: stream large side row-by-row through local hash table (no shuffle)

**SMJ execution:**
- Shuffle both sides by join key (Exchange nodes)
- Sort both sides within each partition
- Merge-join like merge-sort merge step (requires sorted input)

**AQE dynamic switch:** After shuffle map outputs are written, AQE checks actual sizes. If the smaller shuffle output is under `spark.sql.adaptive.autoBroadcastJoinThreshold`, Spark replaces the planned SMJ with BHJ without re-running upstream stages.

### Spark UI — Join Strategy Signals

- **SQL tab > physical plan:** Look for `BroadcastHashJoin`, `SortMergeJoin`, `ShuffledHashJoin`, `BroadcastNestedLoopJoin` operator names.
- **BHJ performance:** Check `BroadcastExchange` duration — if it takes > 10s, the broadcast payload is large; consider raising `spark.sql.broadcastTimeout` (default 300s).
- **SMJ skew:** In stage details, if max task time >> median task time, one join key partition is skewed. Enable `spark.sql.adaptive.skewJoin.enabled=true`.
- **SHJ spill:** ShuffledHashJoin can spill if the build side doesn't fit in memory — look for `spill (disk)` metric on the SHJ stage.
- **AQE plan changes:** In SQL tab, `AdaptiveSparkPlan` node shows `isFinalPlan=true` when AQE finished adapting.

In [ ]:
def join_strategy_advisor(left_size_mb, right_size_mb, has_equi_join, broadcast_threshold_mb=10):
    smaller = min(left_size_mb, right_size_mb)
    larger = max(left_size_mb, right_size_mb)
    print(f'=== Join Strategy Advisor ===')
    print(f'  Left: {left_size_mb} MB, Right: {right_size_mb} MB')
    print(f'  Equi-join: {has_equi_join}, Broadcast threshold: {broadcast_threshold_mb} MB')

    if smaller <= broadcast_threshold_mb and has_equi_join:
        strategy = 'BroadcastHashJoin'
        reason = f'Smaller side ({smaller}MB) under broadcast threshold ({broadcast_threshold_mb}MB)'
        risk = 'OOM on driver/executor if broadcast data is larger than estimated'
    elif not has_equi_join:
        strategy = 'BroadcastNestedLoopJoin'
        reason = 'Non-equi join — no hash/sort strategy available'
        risk = f'O(n*m) complexity = {left_size_mb * right_size_mb} MB^2 equivalent — use only for small tables'
    elif smaller < 200 and larger > 1000:
        strategy = 'Consider broadcast hint — or AQE dynamic broadcast'
        reason = f'Significant size asymmetry: {smaller}MB vs {larger}MB'
        risk = 'May need to raise autoBroadcastJoinThreshold or use broadcast() hint explicitly'
    else:
        strategy = 'SortMergeJoin'
        reason = 'Both sides large, equi-join — default safe choice'
        risk = 'Two shuffles + two sorts — ensure sufficient shuffle partitions'

    print(f'  Recommended: {strategy}')
    print(f'  Reason     : {reason}')
    print(f'  Risk       : {risk}')
    return strategy

join_strategy_advisor(500_000, 200, True)
print()
join_strategy_advisor(500_000, 200, True, broadcast_threshold_mb=250)
print()
join_strategy_advisor(1_000, 800, False)

### Results Deep Dive — Join Strategies

The `explain()` output for each join type shows the physical operator and its children.

Key patterns to verify the correct strategy was applied:
- BHJ: `BroadcastHashJoin [LicenseType]` with a `BroadcastExchange` child on one side
- SMJ: `SortMergeJoin [LicenseType]` with two `Exchange hashpartitioning(...)` children
- SHJ: `ShuffledHashJoin [LicenseType]` — one side is the build (hash table), other is probe
- BNLJ: `BroadcastNestedLoopJoin` with `BroadcastExchange` — appears for cross joins

The `join_strategy_advisor` is a heuristic tool for pre-flight cluster planning. In production, always validate the actual plan with `explain()` after writing the query.

### Expert Best Practices — Join Strategies

- Set `spark.sql.autoBroadcastJoinThreshold=-1` to disable auto-broadcast in environments where driver memory is constrained, then use explicit `broadcast()` hints only where you have verified size.
- Enable AQE (`spark.sql.adaptive.enabled=true`) in Spark 3.x — it eliminates most manual join tuning by switching strategies at runtime.
- For skewed joins, use AQE skew join handling: `spark.sql.adaptive.skewJoin.enabled=true` and tune `spark.sql.adaptive.skewJoin.skewedPartitionFactor`.
- Anti-pattern: broadcasting a table without checking its runtime size — the size estimate at planning time may be wrong for views or after filters.
- Anti-pattern: using BNLJ for a large cross join — it is `O(n*m)` and will run for hours or OOM.
- Use `spark.sql.join.preferSortMergeJoin=false` to allow ShuffledHashJoin when the build side fits comfortably in memory — SHJ avoids the sort overhead of SMJ.
- For multi-way joins, ensure the smallest table is broadcast first — subsequent joins then operate on already-reduced result sets.
- Monitor `BroadcastExchange` serialization time in UI — if it is slow, the broadcast payload may be larger than estimated due to complex nested types.

In [ ]:
# Production join pipeline with strategy logging
def optimized_join(left, right, join_key, join_type='inner', force_broadcast_right=False):
    from pyspark.sql.functions import broadcast as bc
    from io import StringIO
    import sys, re

    if force_broadcast_right:
        joined = left.join(bc(right), join_key, join_type)
        strategy_hint = 'BroadcastHashJoin (forced)'
    else:
        joined = left.join(right, join_key, join_type)
        strategy_hint = 'auto (AQE or threshold-based)'

    buf = StringIO()
    old = sys.stdout; sys.stdout = buf
    joined.explain()
    sys.stdout = old
    plan = buf.getvalue()

    detected = 'Unknown'
    for strat in ['BroadcastHashJoin', 'SortMergeJoin', 'ShuffledHashJoin', 'BroadcastNestedLoop']:
        if strat in plan:
            detected = strat
            break

    print(f'Join key: {join_key}, Type: {join_type}')
    print(f'Hint: {strategy_hint}')
    print(f'Detected strategy: {detected}')
    return joined

result = optimized_join(left_df, license_df, 'LicenseType', force_broadcast_right=True)
print(f'Row count: {result.count()}')

### Practice Exercises — Join Strategies

1. Create `left = spark.range(1_000_000)` and `right = spark.range(50)` (both with a key column). Run the join with no hints and observe the strategy. Then raise `spark.sql.autoBroadcastJoinThreshold` to force BHJ without a hint. Measure the speedup.
2. Simulate a skewed join: create a DataFrame where 80% of rows have `key=1`. Join it with a lookup table. Observe the skew in the stage task duration chart (via UI or by measuring individual partition sizes). Then enable AQE skew join and compare.
3. Force a BroadcastNestedLoopJoin by writing a non-equi join: `left.join(right, left.val > right.threshold)`. Confirm BNLJ in `explain()`. Time it on small data (1K x 1K) and extrapolate what would happen at 1M x 1M.

## Topic 5 — Cluster Sizing and Capacity Planning

Executor configuration is one of the most impactful tuning levers — wrong sizing causes OOM, GC storms, or underutilization.

| Config | Recommended | Reason |
|---|---|---|
| spark.executor.cores | 5 | Balances parallelism vs HDFS connection overhead |
| spark.executor.memory | (node_mem / executors_per_node) * 0.9 | Leave 10% for OS |
| spark.executor.memoryOverhead | max(384MB, memory * 0.1) | JVM overhead, Python worker, native libs |
| spark.driver.memory | 2–8g depending on collect() size | Must hold broadcast tables + result data |
| spark.executor.instances | (total_cores / 5) | Use all available cores |

### Business Scenario

You are provisioning a new YARN cluster for daily batch jobs processing 10TB of Parquet data.
Nodes: 20 nodes, each with 64 cores and 256GB RAM.
Jobs: 5 concurrent Spark applications.

**Architecture Challenge:** Calculate the optimal executor configuration. Why is 5 cores per executor the 'sweet spot'? What is the memoryOverhead formula and why does it matter on Kubernetes?

In [ ]:
def design_cluster(data_tb, daily_jobs, node_count=20, cores_per_node=64, ram_gb_per_node=256,
                   os_reserve_gb=10, overhead_fraction=0.1):
    usable_ram_gb = ram_gb_per_node - os_reserve_gb
    cores_per_exec = 5  # sweet spot
    executors_per_node = cores_per_node // cores_per_exec
    exec_memory_gb = usable_ram_gb // executors_per_node
    overhead_gb = max(0.375, exec_memory_gb * overhead_fraction)
    heap_gb = exec_memory_gb - overhead_gb
    total_executors = node_count * executors_per_node
    executors_per_job = total_executors // daily_jobs

    # Shuffle partition sizing: 128MB per partition is ideal
    data_mb = data_tb * 1024 * 1024
    shuffle_partitions = max(200, int(data_mb / 128))

    config = {
        'spark.executor.cores': cores_per_exec,
        'spark.executor.memory': f'{int(heap_gb)}g',
        'spark.executor.memoryOverhead': f'{int(overhead_gb * 1024)}m',
        'spark.executor.instances': executors_per_job,
        'spark.driver.memory': '4g',
        'spark.sql.shuffle.partitions': shuffle_partitions,
        'spark.dynamicAllocation.enabled': True,
        'spark.dynamicAllocation.minExecutors': 2,
        'spark.dynamicAllocation.maxExecutors': total_executors,
    }

    print(f'=== Cluster Design: {data_tb}TB data, {daily_jobs} concurrent jobs ===')
    print(f'  Nodes: {node_count} x ({cores_per_node} cores, {ram_gb_per_node}GB RAM)')
    print(f'  Executors per node: {executors_per_node}')
    print(f'  Total executors   : {total_executors}')
    print(f'  Per-job executors : {executors_per_job}')
    print(f'  Executor heap     : {int(heap_gb)}g')
    print(f'  Memory overhead   : {int(overhead_gb*1024)}m')
    print(f'  Shuffle partitions: {shuffle_partitions}')
    print()
    print('Recommended spark-submit config:')
    for k, v in config.items():
        print(f'  --conf {k}={v}')
    return config

cfg = design_cluster(data_tb=10, daily_jobs=5)

### Expert Interview Questions — Cluster Sizing

1. Why is 5 cores per executor considered the sweet spot? What happens with 1 core (tiny executors) and with 30 cores (fat executors)?
2. Walk through the exact formula for `spark.executor.memory` given node hardware specs. What fraction does YARN's `yarn.nodemanager.resource.memory-mb` play?
3. What is `spark.executor.memoryOverhead` and what consumes it? Why is it especially important on Kubernetes vs YARN?
4. Compare YARN dynamic allocation vs Kubernetes dynamic allocation. What are the pitfalls of dynamic allocation with Structured Streaming?
5. What is `spark.memory.fraction` and `spark.memory.storageFraction`? How do they determine how much memory is available for execution vs caching?
6. How do you size `spark.sql.shuffle.partitions` for a given data volume? What is the target partition size?
7. What is the '3 executor rule' for fat executor anti-pattern? Why does HDFS throughput degrade with many concurrent threads per executor?
8. On Kubernetes, what happens if a container exceeds its memory limit? How does `memoryOverhead` prevent OOMKilled pods?

### Sizing Models Deep Dive

**Tiny executor model (anti-pattern):**
- 1 core, 2GB RAM per executor
- Many executors share the same node
- Cannot benefit from in-JVM data sharing across tasks
- High JVM process overhead ratio

**Fat executor model (anti-pattern):**
- 30+ cores, 200GB RAM per executor
- HDFS client performance degrades beyond ~5 concurrent threads
- GC pauses affect all threads in the process simultaneously
- Single point of failure for all tasks on that executor

**Balanced model (recommended):**
- 5 cores, (node_ram / executors_per_node) * 0.9 GB
- HDFS: 5 concurrent reads = good throughput without contention
- GC pause scope limited to 5 tasks
- 1 core left per node for YARN/OS daemons

**Memory overhead sources:**
- JVM internals (JIT, class loading)
- Python worker process memory (PySpark UDFs spawn subprocesses)
- Native libraries (OpenSSL, Hadoop native, Arrow)
- Off-heap memory allocations (if `spark.memory.offHeap.enabled=true`)

### Spark UI — Cluster Sizing Signals

- **Executors tab:** Check `GC Time` column. If any executor has GC > 10% of task time, executor memory is too small.
- **Storage tab:** If cached RDDs/DataFrames show `DISK_ONLY` fractions, storage memory is being evicted by execution — consider increasing `spark.memory.storageFraction`.
- **Stages tab:** If most stages show `Max task time >> Median task time` (ratio > 3x), you have data skew — not a sizing problem.
- **Executors tab > Failed tasks:** OOM errors appear here. Look for `ExecutorLostFailure` or `java.lang.OutOfMemoryError`.
- **Environment tab:** Verify your `spark.executor.memory` and `spark.executor.memoryOverhead` are set as intended.

In [ ]:
def validate_cluster_config(spark_conf_dict):
    import re
    warnings = []
    info = []

    cores = int(spark_conf_dict.get('spark.executor.cores', 5))
    mem_str = spark_conf_dict.get('spark.executor.memory', '4g')
    mem_gb = int(re.sub(r'[gG]', '', mem_str))
    overhead_str = spark_conf_dict.get('spark.executor.memoryOverhead', '384m')
    overhead_mb = int(re.sub(r'[mM]', '', overhead_str)) if 'm' in overhead_str.lower() else int(re.sub(r'[gG]', '', overhead_str)) * 1024
    shuffle_parts = int(spark_conf_dict.get('spark.sql.shuffle.partitions', 200))

    if cores > 10:
        warnings.append(f'executor.cores={cores} is too high (sweet spot=5) — HDFS throughput will degrade')
    if cores == 1:
        warnings.append('executor.cores=1 (tiny executor anti-pattern) — no intra-executor parallelism')
    if mem_gb < 2:
        warnings.append(f'executor.memory={mem_gb}g is too low — likely OOM on any real workload')
    if overhead_mb < 384:
        warnings.append(f'memoryOverhead={overhead_mb}m < minimum 384m — risky on Kubernetes (OOMKilled)')
    if shuffle_parts < 10:
        warnings.append(f'shuffle.partitions={shuffle_parts} very low — bottleneck on large aggregations')
    if shuffle_parts > 2000:
        warnings.append(f'shuffle.partitions={shuffle_parts} very high — too many small tasks, scheduling overhead')

    info.append(f'Cores per executor: {cores}')
    info.append(f'Heap per executor : {mem_gb}g')
    info.append(f'Overhead          : {overhead_mb}m')
    info.append(f'Shuffle partitions: {shuffle_parts}')

    print('=== Cluster Config Validation ===')
    for i in info: print(f'  INFO   : {i}')
    for w in warnings: print(f'  WARNING: {w}')
    print(f'  Status : {"PASS" if not warnings else f"{len(warnings)} warning(s)"}')

validate_cluster_config(cfg)

### Results Deep Dive — Cluster Sizing

The `design_cluster()` output provides a starting point. Real-world adjustments needed:

- **Shuffle partitions:** `data_mb / 128` is a guideline. After running the first job, check average partition size in the SQL tab. Aim for 100–200MB per shuffle partition for large aggregations.
- **Dynamic allocation:** Useful for mixed workloads but has a warm-up penalty. For latency-sensitive jobs, prefer static allocation.
- **Memory overhead on Kubernetes:** Container memory limit = `executor.memory + memoryOverhead`. If the process exceeds the container limit, Kubernetes sends SIGKILL — no graceful shutdown, tasks are lost.
- **YARN reserve:** YARN's `yarn.nodemanager.resource.memory-mb` must account for all containers plus OS. Typical formula: `(node_ram - 1GB) * 0.9`.

### Expert Best Practices — Cluster Sizing

- Always leave 1 core per node for YARN NodeManager, HDFS DataNode, and OS processes.
- For PySpark workloads, add an extra 512MB to `memoryOverhead` per executor to account for Python subprocess memory.
- Anti-pattern: setting `spark.executor.instances` to a fixed large number without dynamic allocation — wastes resources during idle periods.
- Anti-pattern: using default `spark.sql.shuffle.partitions=200` on a 10TB dataset — results in 50GB+ per partition causing spill.
- For Kubernetes, set `spark.kubernetes.executor.request.cores` = `spark.executor.cores` for accurate scheduling.
- Monitor `spark.executor.memoryOverhead` usage via `container_memory_usage_bytes` Prometheus metric on K8s.
- Benchmark with realistic data before finalizing cluster size — synthetic benchmarks (TPC-DS) are a good starting point.
- Use Ganglia, Prometheus, or Spark History Server to build baseline metrics across job types before optimizing.

In [ ]:
# Production capacity planning report
def capacity_planning_report(data_sizes_tb, node_count, cores_per_node, ram_gb_per_node):
    print(f'=== Capacity Planning Report ===')
    print(f'Hardware: {node_count} nodes, {cores_per_node} cores, {ram_gb_per_node}GB RAM each')
    print()
    for tb in data_sizes_tb:
        cores_per_exec = 5
        executors_per_node = (cores_per_node - 1) // cores_per_exec
        usable_ram = ram_gb_per_node - 10
        exec_mem = usable_ram // executors_per_node
        overhead = max(0.375, exec_mem * 0.1)
        heap = exec_mem - overhead
        total_exec = node_count * executors_per_node
        shuffle_parts = max(200, int(tb * 1024 * 1024 / 128))
        print(f'  Data: {tb}TB => shuffle.partitions={shuffle_parts}, executor.memory={int(heap)}g, total_executors={total_exec}')

capacity_planning_report([1, 10, 50, 100], 20, 64, 256)

### Practice Exercises — Cluster Sizing

1. You have 10 nodes with 32 cores and 128GB RAM each. Design 3 configurations: tiny (1 core/exec), fat (30 cores/exec), and balanced (5 cores/exec). For each, calculate total executor count, memory per executor, and expected memoryOverhead. Use `validate_cluster_config()` to check each.
2. A job runs in 30 minutes with static allocation of 20 executors. With dynamic allocation enabled (`minExecutors=2, maxExecutors=50`), the job takes 35 minutes but costs 40% less in cloud compute. Write a cost-benefit analysis function: `def cost_benefit(static_exec, dynamic_min, dynamic_max, price_per_exec_hour)`.
3. Simulate shuffle partition sizing: use `spark.range(10_000_000)` grouped by `id % N` where N is 10, 100, 1000. Measure execution time and partition sizes for each. Plot the results and determine the optimal partition count.

## Topic 6 — Executor Tuning and GC

Spark's unified memory model and JVM garbage collection are the two most common sources of executor-level performance problems.

| Memory Region | Config | Default | Role |
|---|---|---|---|
| Reserved memory | Fixed 300MB | 300MB | Spark internal objects |
| Spark memory | (heap - 300MB) * memoryFraction | 60% of usable | Execution + Storage |
| Execution memory | Spark memory * (1 - storageFraction) | Dynamic | Shuffles, sorts, joins |
| Storage memory | Spark memory * storageFraction | Dynamic | Cached RDDs/DFs |
| User memory | (heap - 300MB) * (1 - memoryFraction) | 40% of usable | User data structures |

### Business Scenario

An executor crashes with `java.lang.OutOfMemoryError: GC overhead limit exceeded` during a wide sort operation.
The executor heap is 8GB. The job has both cached DataFrames and a large `sortWithinPartitions`.

**Architecture Challenge:** How do you diagnose whether the OOM is caused by execution memory, storage memory, or user memory? What are the tuning levers in order of preference?

In [ ]:
# Demonstrate memory pressure: large sort on synthetic data
# Uses spark.range for large-scale synthetic data
mem_test_df = spark.range(1_000_000).selectExpr(
    'id',
    'id % 1000 as sort_key',
    'cast(rand() * 10000 as double) as value1',
    'cast(rand() * 10000 as double) as value2',
    'concat("prefix_", cast(id as string)) as str_col'
)

print('=== Sort operation plan (shows sort + exchange) ===')
sorted_df = mem_test_df.sortWithinPartitions('sort_key', 'value1')
sorted_df.explain()

print('\n=== Cache + sort interaction ===')
cached_df = mem_test_df.cache()
cached_df.count()  # trigger caching
print(f'Cached partitions: {spark.sparkContext.statusTracker().getExecutorInfos()}')

# Show memory config
print('\nMemory configs:')
for c in ['spark.memory.fraction', 'spark.memory.storageFraction',
          'spark.executor.memory', 'spark.executor.memoryOverhead']:
    print(f'  {c} = {spark.conf.get(c, "default")}')

### Expert Interview Questions — Executor Tuning and GC

1. Describe Spark's unified memory model. What is the difference between execution memory and storage memory? What happens when execution needs more memory than its allocation?
2. What is `spark.memory.fraction` and `spark.memory.storageFraction`? How do they interact to determine how much memory each pool gets?
3. When does execution memory evict storage memory? When does the reverse happen? What gets written to disk during eviction?
4. What JVM GC collector does Spark recommend and why? Compare G1GC vs CMS vs ZGC for Spark workloads.
5. What is `spark.executor.memoryOverhead` and what heap allocations does it cover that are NOT part of the JVM heap?
6. When should you enable `spark.memory.offHeap.enabled=true`? What operations benefit most from off-heap memory?
7. What causes `GC overhead limit exceeded` specifically? How do you tell if it is execution memory vs storage memory causing pressure?
8. What is the `spark.storage.memoryMapThreshold` setting and when does it matter for shuffle performance?

### Memory Model Deep Dive

**Unified Memory Manager (Spark 1.6+):**
```
Total JVM Heap
  └─ Reserved Memory (300MB fixed)
  └─ Usable = Heap - 300MB
       └─ Spark Memory = Usable * spark.memory.fraction (default 0.6)
       |    └─ Storage Pool: starts at Usable * fraction * storageFraction (0.5)
       |    └─ Execution Pool: starts at Usable * fraction * (1 - storageFraction)
       |    └─ Pools are soft boundaries — either can borrow from the other
       └─ User Memory = Usable * (1 - spark.memory.fraction) (default 0.4)
```

**Eviction rules:**
- Execution can evict storage if it needs memory (storage data written to disk or dropped)
- Storage CANNOT evict execution memory (execution is in-progress, cannot be interrupted)
- This asymmetry means heavy execution workloads can cause cached DataFrames to be evicted to disk

**GC tuning for G1GC (recommended):**
`-XX:+UseG1GC -XX:+UnlockDiagnosticVMOptions -XX:G1SummarizeRSetStatsPeriod=1 -XX:InitiatingHeapOccupancyPercent=35`

### Physical Plan — Memory Indicators

Plans that are most memory-intensive (watch in UI):

- `Sort` — buffers an entire partition in execution memory before outputting
- `HashAggregate` — builds hash map in execution memory (can spill if too large)
- `SortMergeJoin` — sorts both sides, buffers join partition
- `BroadcastExchange` — entire table held in driver + executor memory
- `WindowExec` — buffers all rows for a window frame

When you see `spill (memory)` and `spill (disk)` metrics on these operators in the UI, execution memory was exhausted.

### Spark UI — Memory and GC Signals

- **Executors tab > GC Time:** Sum across all executors. GC% = GC Time / (Task Time + GC Time). Alert at 5%+.
- **Stage detail > spill (disk):** Any non-zero value means execution memory was exhausted — increase executor memory or shuffle partitions.
- **Storage tab > Fraction Cached:** If < 100%, some partitions were evicted. Execution pressure is competing with storage.
- **Executors tab > Peak Execution Memory:** High values near executor memory limit indicate risk of OOM.
- **Environment tab:** Verify G1GC flags are set: `-XX:+UseG1GC` should appear in JVM options.
- **Logs for OOM:** `java.lang.OutOfMemoryError: Java heap space` = heap exhausted; `GC overhead limit exceeded` = GC spending > 98% of time with < 2% recovery.

In [ ]:
def validate_memory_config(spark):
    print('=== Executor Memory Configuration Audit ===')
    configs = [
        ('spark.memory.fraction', '0.6', 'Fraction of heap for Spark memory pool'),
        ('spark.memory.storageFraction', '0.5', 'Starting fraction for storage within Spark memory'),
        ('spark.executor.memory', None, 'Total JVM heap per executor'),
        ('spark.executor.memoryOverhead', None, 'Off-heap overhead per executor'),
        ('spark.memory.offHeap.enabled', 'false', 'Off-heap allocation switch'),
        ('spark.memory.offHeap.size', '0', 'Off-heap size if enabled'),
    ]
    for key, expected, desc in configs:
        val = spark.conf.get(key, 'NOT SET')
        flag = ''
        if expected and val != expected and val != 'NOT SET':
            flag = ' <-- non-default'
        print(f'  {key} = {val}{flag}')
        print(f'    ({desc})')

    # Compute effective memory fractions
    frac = float(spark.conf.get('spark.memory.fraction', '0.6'))
    storage_frac = float(spark.conf.get('spark.memory.storageFraction', '0.5'))
    user_frac = 1.0 - frac
    print(f'\n  Effective breakdown (ignoring 300MB reserved):')
    print(f'    Spark memory (exec+storage): {frac*100:.0f}%')
    print(f'    User memory               : {user_frac*100:.0f}%')
    print(f'    Storage starts at         : {frac * storage_frac * 100:.0f}% of heap')
    print(f'    Execution starts at       : {frac * (1-storage_frac) * 100:.0f}% of heap')

validate_memory_config(spark)

### Results Deep Dive — GC and Memory

The memory audit shows that with default settings, out of a 4GB executor heap:
- 300MB is reserved
- ~2.22GB is Spark memory (60% of usable ~3.7GB)
- ~1.48GB is user memory (40%)
- Storage and execution each start at ~1.11GB but can borrow from each other

For a job with both caching and heavy sorts:
1. Raise `spark.memory.storageFraction` to 0.6 if cache hit rate is critical and sort memory is sufficient
2. Raise `spark.memory.fraction` to 0.75 if user code is simple and workload is execution-heavy
3. Add `spark.executor.memoryOverhead` increase for PySpark jobs (Python workers are counted here)

### Expert Best Practices — Executor Tuning

- Start with default memory fractions and tune only after profiling with Spark History Server — premature tuning often makes things worse.
- For sort-heavy workloads, increase `spark.sql.shuffle.partitions` before increasing executor memory — smaller partitions reduce per-task memory pressure.
- Enable off-heap memory for workloads with large binary datasets: `spark.memory.offHeap.enabled=true` and set size to 20-30% of executor memory.
- G1GC settings to add to `spark.executor.extraJavaOptions`: `-XX:+UseG1GC -XX:InitiatingHeapOccupancyPercent=35 -XX:ConcGCThreads=4`.
- Anti-pattern: setting `spark.memory.fraction` to 0.9 — leaves only 10% for user code, causing OOM on complex transformations.
- Anti-pattern: caching massive DataFrames with `MEMORY_AND_DISK` without checking available storage pool size — leads to constant eviction and disk I/O.
- For Python UDFs: each PySpark task spawns a Python subprocess. The subprocess memory comes from `memoryOverhead`, not the JVM heap.
- Use `df.unpersist()` explicitly after a cached DataFrame is no longer needed — Spark's LRU eviction is not immediate and wastes storage memory.

In [ ]:
# Production memory pressure test + spill detection
def memory_pressure_test(spark, n_rows=500_000):
    test = spark.range(n_rows).selectExpr(
        'id',
        'id % 10000 as sort_key',
        'cast(rand() * 1000000 as double) as val'
    )
    t0 = time.time()
    result = test.sortWithinPartitions('sort_key').agg(_sum('val')).collect()
    t1 = time.time()
    print(f'Memory pressure test: {n_rows:,} rows sorted + aggregated in {t1-t0:.2f}s')
    print(f'Result: {result[0][0]:.2f}')
    print('Check Spark UI Stages tab for spill (memory) and spill (disk) metrics')
    return t1 - t0

memory_pressure_test(spark)

### Practice Exercises — Executor Tuning

1. Run `memory_pressure_test()` with n_rows = 100K, 500K, 1M, 2M. Record execution time and check for disk spill in each run. At what row count does spill first appear for your executor configuration?
2. Cache a 500K row DataFrame with `MEMORY_AND_DISK`. Then run a heavy sort on a different DataFrame that requires more memory than available. Use the Storage UI tab to observe whether cached partitions are evicted. Try increasing `spark.memory.storageFraction` to 0.7 and repeat.
3. Add G1GC options to `spark.executor.extraJavaOptions` and compare GC time before/after across 5 runs of the same aggregation job. Build a function `def gc_benchmark(spark, n_runs)` that returns average/min/max GC time percentage.

## Topic 7 — End-to-End Pipeline Optimization

A complete ETL pipeline requires integrating all optimization techniques: predicate pushdown, broadcast joins, partition pruning, sorted writes, and AQE.

| Optimization Layer | Technique | Impact |
|---|---|---|
| Read layer | Early filter + column pruning | Reduces scan I/O |
| Join layer | Broadcast small dims, sort-merge large facts | Eliminates shuffle for small tables |
| Aggregate layer | Partial aggregation + AQE | Reduces shuffle data volume |
| Write layer | partitionBy + sortWithinPartitions | Enables partition pruning in downstream reads |
| Global | AQE enabled | Runtime plan adjustments |

### Business Scenario

A daily ETL job reads Cabs.csv, joins to a license lookup, computes a window rank of cabs by VehicleYear within each LicenseType, then writes to Parquet.
The naive version runs in 8 minutes. The optimized version should run under 2 minutes.

**Architecture Challenge:** Walk through every optimization decision: why to push filters before the join, why to broadcast the lookup, why to sort before the window, and why to partition the Parquet output.

In [ ]:
import tempfile, os
TMP_OUT = os.path.join(tempfile.gettempdir(), 'spark_etl_output')

# === NAIVE PIPELINE (no optimizations) ===
def naive_pipeline(cabs, license_df, output_path):
    # No early filter, no broadcast hint, no partitioning
    joined = cabs.join(license_df, 'LicenseType', 'left')
    w = Window.partitionBy('LicenseType').orderBy(col('VehicleYear').desc())
    ranked = joined.withColumn('year_rank', rank().over(w))
    aggregated = ranked.groupBy('LicenseType', 'ServiceDesc').agg(
        count('*').alias('total_cabs'),
        avg('VehicleYear').alias('avg_year'),
        _max('VehicleYear').alias('max_year')
    )
    aggregated.write.mode('overwrite').parquet(output_path + '_naive')
    return aggregated

# Create license lookup
license_data2 = [
    ('TAXI', 'Taxi Service'),
    ('LIMOUSINE', 'Limo Service'),
    ('BUS', 'Bus Service'),
    ('CHARTER', 'Charter Service'),
    ('LIVERY', 'Livery Service')
]
lic2 = spark.createDataFrame(license_data2, ['LicenseType', 'ServiceDesc'])

t0 = time.time()
naive_result = naive_pipeline(cabs, lic2, TMP_OUT)
t_naive = time.time() - t0
print(f'Naive pipeline: {t_naive:.2f}s, rows: {naive_result.count()}')
naive_result.explain()

### Expert Interview Questions — Pipeline Optimization

1. What is the correct order of operations in an optimized ETL pipeline? Why should filters and column selections come before joins?
2. When should you materialize (persist/cache) an intermediate result in a multi-stage pipeline? What are the risks of over-caching?
3. What is the difference between `df.checkpoint()` and `df.persist()`? When would you use checkpointing in a pipeline?
4. How does `partitionBy()` in a write operation affect downstream query performance? What is partition pruning?
5. Explain the trade-off between shuffle partition count and output file count when using `repartition()` before a write.
6. What is `sortWithinPartitions()` vs `orderBy()`? When does using `orderBy()` cause a full global sort shuffle?
7. How does AQE's `coalesceShufflePartitions` feature help pipeline performance? What is `spark.sql.adaptive.coalescePartitions.initialPartitionNum`?
8. What is the `spark.sql.files.maxPartitionBytes` config and how does it affect read-side parallelism?

### Pipeline Optimization Internals

**Optimization order of impact (highest first):**
1. Push filters before joins — most impactful, reduces rows flowing into expensive operations
2. Column pruning — reduces bytes read from Parquet/ORC (only load needed columns)
3. Broadcast small dimensions — eliminates shuffle for small-large joins
4. Partition pruning on read — skip entire files/directories
5. AQE coalesce empty partitions — reduce task overhead on skewed outputs
6. sortWithinPartitions before write — enables merge-sort in downstream SMJ without re-sorting

**Checkpointing use cases:**
- Very long lineage chains (100+ transformations) — checkpointing truncates the DAG
- Iterative algorithms (graph, ML) — prevents recomputation of base data
- Fault tolerance across stage boundaries in streaming

**partitionBy write strategy:**
- Each distinct value of the partition column becomes a directory
- Downstream filters on the partition column skip entire directories
- Too many distinct values = too many small files (the 'small files problem')

### Physical Plan Comparison — Naive vs Optimized

**Naive plan (key problems):**
- Join appears before filter — all rows (including inactive cabs) enter the join
- No BroadcastExchange for the small license table — full SortMergeJoin with two shuffles
- Window function `rank()` applied to full unfiltered dataset
- Output not partitioned — downstream reads scan all files

**Optimized plan (after fixes):**
- Filter `Active=YES AND VehicleYear > 2010` pushed before join — smaller left side
- `broadcast(lic2)` — BroadcastHashJoin replaces SortMergeJoin (one less shuffle)
- Window operates on already-filtered smaller dataset
- `partitionBy('LicenseType')` — downstream queries filter by license skip directories
- `sortWithinPartitions('VehicleYear')` — prepares data for efficient downstream SMJ

### Spark UI — Pipeline Health Checklist

- **Jobs tab:** Count total jobs triggered. Each `action` (count, write, collect) = 1 job. Unexpected extra jobs = lazy evaluation surprise.
- **Stages tab:** Look for stages with very low `Input` but high `Shuffle Write` — indicates a stage that produces more data than it reads (possible fan-out/explode).
- **SQL tab > physical plan:** Verify BroadcastExchange is present for small dimensions. Verify FileScan shows `PushedFilters`.
- **Writing stages:** Check `Output` size. If writing 1000 small files, add a `coalesce()` or `repartition()` before write.
- **AQE plan:** Confirm `coalesceShufflePartitions` fired — look for `CustomShuffleReader` with merged partitions in the SQL plan.

In [ ]:
# === OPTIMIZED PIPELINE ===
def optimized_pipeline(cabs, license_df, output_path):
    # 1. Early filter + column pruning
    filtered = (cabs
        .filter(col('Active') == 'YES')
        .filter(col('VehicleYear') > 2010)
        .select('CabNumber', 'Name', 'LicenseType', 'VehicleYear'))

    # 2. Broadcast the small lookup
    joined = filtered.join(broadcast(license_df), 'LicenseType', 'left')

    # 3. Window on filtered+joined data (smaller)
    w = Window.partitionBy('LicenseType').orderBy(col('VehicleYear').desc())
    ranked = joined.withColumn('year_rank', rank().over(w))

    # 4. Aggregation
    aggregated = ranked.groupBy('LicenseType', 'ServiceDesc').agg(
        count('*').alias('total_cabs'),
        avg('VehicleYear').alias('avg_year'),
        _max('VehicleYear').alias('max_year')
    )

    # 5. Sorted write with partition by
    (aggregated
        .sortWithinPartitions('LicenseType', 'avg_year')
        .write
        .mode('overwrite')
        .partitionBy('LicenseType')
        .parquet(output_path + '_optimized'))
    return aggregated

t0 = time.time()
opt_result = optimized_pipeline(cabs, lic2, TMP_OUT)
t_opt = time.time() - t0
print(f'Optimized pipeline: {t_opt:.2f}s, rows: {opt_result.count()}')
opt_result.explain()

if t_naive > 0:
    print(f'\nSpeedup: {t_naive/t_opt:.2f}x')

### Results Deep Dive — Pipeline

The optimized pipeline should show measurable improvement even on the small Cabs dataset. On production-scale data, the improvements compound:

- **Early filter** on a 1TB table with 30% active rows reduces join input by 70%
- **Broadcast join** on a 50-row lookup eliminates 2 shuffle stages (~30s saved on large clusters)
- **partitionBy on write** enables partition pruning — a downstream query filtering one LicenseType reads 1/N of the data
- **sortWithinPartitions** is local (no shuffle) and prepares data for efficient downstream SMJ without a sort stage

On small test data, the difference may be under 1 second. Always validate optimizations at production scale.

### Expert Best Practices — Pipeline

- Apply filters and projections as early in the pipeline as possible — they reduce data volume for every subsequent operation.
- Do not use `orderBy()` (global sort) unless you need a globally sorted output — use `sortWithinPartitions()` for write optimization.
- When writing Parquet, target 128–512MB per file. Use `coalesce(N)` before write to control file count.
- Anti-pattern: calling `.count()` as a pipeline health check inside a production job — it triggers a full scan and adds latency.
- Anti-pattern: caching every intermediate DataFrame 'just in case' — each cache costs storage memory and can evict other cached data.
- Checkpoint long pipelines (50+ transformations) to break lineage and improve fault recovery time.
- Use `.explain('formatted')` to diff naive vs optimized plans side-by-side — look for BroadcastExchange and PushedFilters.
- Validate partitioned Parquet output by reading it back with a partition filter and confirming `FileScan` shows partition pruning.

### Practice Exercises — Pipeline

1. Add a `df.cache()` call between the filter and the join in the optimized pipeline. Run `explain()` on the post-cache steps. Does the plan change? Run the pipeline 3 times and compare: first call (cold cache), second call (warm cache), third call. Is the cache hit worth the memory cost?
2. Add a `checkpoint()` call after the window function step. Configure `spark.sparkContext.setCheckpointDir('/tmp/checkpoints')`. Compare the `explain()` output before and after checkpoint — the lineage should be truncated. Measure the overhead of checkpointing vs the benefit for fault recovery.
3. Experiment with write partitioning: write the output with `partitionBy('LicenseType')` and then with `partitionBy('LicenseType', 'VehicleYear')`. Compare the number of output files, total output size, and the read performance for a single-LicenseType filter query.

## Topic 8 — Spark UI Deep Dive

The Spark UI is the primary diagnostic tool for understanding and optimizing Spark job performance.

| UI Tab | Key Information | Alert Conditions |
|---|---|---|
| Jobs | DAG, job duration, success/failure | Failed jobs, unusually long jobs |
| Stages | Task metrics, shuffle read/write, spill | Max task >> median task (skew), spill > 0 |
| Storage | Cached RDD/DF size, fraction cached | Low fraction cached (eviction) |
| Environment | Spark version, all configs, JVM flags | Non-default configs, wrong GC flags |
| Executors | GC time, task counts, OOM | GC% > 5%, failed tasks |
| SQL/DataFrame | Physical plan, per-operator metrics | Missing BroadcastExchange, unsupported pushdown |

### Business Scenario

A production Spark job is taking 3x longer than expected. The SLA is 30 minutes but it is running 90 minutes.
You have read access to the Spark History Server. Walk through each UI tab and explain exactly what you look for and what each finding implies.

**Architecture Challenge:** Given only Spark UI data (no code access), identify the root cause from a list of symptoms and prescribe the exact config change or code fix needed.

In [ ]:
# Run a representative join query to populate UI metrics
from pyspark.sql.functions import broadcast

# Create datasets for a meaningful UI demo
facts = spark.range(500_000).selectExpr(
    'id',
    'id % 100 as cab_group',
    'cast(rand() * 5000 as double) as trip_value',
    'id % 10 as zone_id'
)

dims = spark.range(10).selectExpr(
    'id as zone_id',
    'concat("Zone_", cast(id as string)) as zone_name'
)

# Pipeline with join + window + agg
w = Window.partitionBy('zone_id').orderBy('trip_value')
result = (facts
    .join(broadcast(dims), 'zone_id', 'left')
    .withColumn('zone_rank', rank().over(w))
    .filter(col('zone_rank') <= 100)
    .groupBy('zone_name')
    .agg(
        count('*').alias('top_trips'),
        _sum('trip_value').alias('total_value'),
        avg('trip_value').alias('avg_value')
    )
    .orderBy('total_value', ascending=False)
)

result.show()
print('\n=== Physical Plan ===')
result.explain('formatted')

### Expert Interview Questions — Spark UI

1. What does a high GC time percentage on the Executors tab tell you? What are the 3 most common causes and their fixes?
2. How do you identify data skew from the Spark UI without looking at the code? Which tab and which metric column reveals it?
3. What is the Spark shuffle service and why does it exist? How do you verify it is enabled in the Environment tab?
4. Explain the DAG visualization in the Jobs tab. What does a wide dependency vs narrow dependency look like and why does it matter?
5. In the SQL tab, what metrics appear on a `SortMergeJoin` node? How do you tell if the join is spilling to disk?
6. What causes high task deserialization time? When does it indicate a problem and when is it expected?
7. How do you use the Spark UI to verify that AQE changed the execution plan at runtime? Where exactly do you see the AQE-modified plan?
8. What is the `Shuffle Read Blocked Time` metric in stage details? What infrastructure change fixes a high value?

### Spark UI Internals Deep Dive

**Jobs tab anatomy:**
- Each `action` (collect, count, write, show) triggers one job
- Stages within a job are separated by shuffle boundaries (Exchange operators)
- Stage dependencies shown as DAG — wide dependency = shuffle, narrow = pipeline

**Stages tab — key columns:**
- `Input`: bytes read from HDFS/S3/local
- `Shuffle Read` / `Shuffle Write`: bytes crossing the shuffle boundary
- `Spill (Memory)` / `Spill (Disk)`: execution memory overflow
- `Duration` percentiles: look for max >> 75th percentile (skew signal)

**Skew detection:**
- `Max task duration` / `Median task duration` > 3x = significant skew
- `Max bytes read` / `Median bytes read` > 3x = partition size skew

**SQL tab metrics on operators:**
- `number of output rows` — output size after operator
- `time in WholeStageCodegen` — time in codegen stage
- `spill size` on Sort/HashAgg — confirms execution memory exhaustion

### Physical Plan — UI Correlation

The SQL tab's physical plan nodes map 1:1 to stage boundaries:

- Each `Exchange` node = 1 stage boundary = 1 shuffle
- Operator metrics (rows in/out, time) appear when you click node in UI
- `WholeStageCodegen` blocks show as a single fused node in UI
- `AQE` wrapper: `AdaptiveSparkPlan` is the root — click 'Final Plan' to see runtime-adjusted plan

**Broadcast metrics in UI:**
- `BroadcastExchange` shows `time to broadcast` — if > 30s, data too large
- `BroadcastHashJoin` shows `number of output rows` — verify join is filtering correctly

In [ ]:
def spark_ui_checklist():
    checklist = [
        ('Jobs Tab', [
            'Is job count reasonable? Each write/count/show = 1 job',
            'Are there failed jobs? Check for retried stages',
            'Duration: any job taking 10x longer than similar jobs?',
        ]),
        ('Stages Tab', [
            'Skew check: Max task time / Median task time > 3x = skew',
            'Spill check: Spill (Disk) > 0 = execution memory exhausted',
            'Shuffle read: any stage reading > 10x its input = explode/join fan-out',
            'Input size: scan stages reading full table? Check for missing predicate pushdown',
        ]),
        ('Storage Tab', [
            'Fraction cached < 100%? Storage memory being evicted by execution',
            'DISK_ONLY partitions? Cache is overflowing to disk',
        ]),
        ('Executors Tab', [
            'GC Time > 5% of task time = memory pressure',
            'Failed tasks > 0 = OOM or executor lost',
            'Peak execution memory near executor memory limit = OOM risk',
        ]),
        ('SQL/DataFrame Tab', [
            'BroadcastExchange present for small dimensions?',
            'PushedFilters in FileScan? If empty, check column types',
            'WholeStageCodegen (*(N)) markers on main operators?',
            'AQE: AdaptiveSparkPlan > Final Plan shows runtime adjustments',
            'Sort/HashAgg spill metric > 0 = execution memory issue',
        ]),
        ('Environment Tab', [
            'spark.sql.adaptive.enabled = true?',
            'spark.sql.shuffle.partitions appropriate for data size?',
            'JVM flags: -XX:+UseG1GC present?',
            'spark.executor.memoryOverhead set?',
        ]),
    ]
    print('=== Spark UI Diagnostic Checklist ===')
    for tab, items in checklist:
        print(f'\n[{tab}]')
        for item in items:
            print(f'  [ ] {item}')

spark_ui_checklist()

### Results Deep Dive — UI Metrics

After running the join + window + aggregation pipeline above, the Spark UI will show:

- **Jobs:** 2-3 jobs (1 for the window/join pipeline, 1 for the final aggregation action, 1 for show())
- **Stages:** Stage with BroadcastExchange should show small `Shuffle Write` — broadcast avoids most shuffling
- **SQL plan:** `BroadcastHashJoin` with `BroadcastExchange` on the dims side; `WindowExec` for rank; final `HashAggregate`
- **WholeStageCodegen:** `*(N)` markers on Range, Filter, Project, and partial HashAggregate stages
- **Executors:** GC should be low (< 2%) on this small dataset

In a real production job, correlate high GC time with stages that have large shuffle read — those stages are building large in-memory structures.

### Expert Best Practices — Spark UI

- Bookmark the Spark History Server URL for your cluster — live UI disappears when the application ends.
- For recurring jobs, compare UI metrics run-over-run using History Server REST API (`/api/v1/applications`) to detect regressions.
- When debugging a slow stage: first check `Max task time / Median task time` for skew. If not skewed, check `Spill (Disk)`. If no spill, check `Shuffle Read` size.
- Anti-pattern: ignoring the SQL tab and only looking at Jobs/Stages — the physical plan in SQL tab is the fastest way to identify missing optimizations.
- Anti-pattern: treating all long-running jobs the same — a 30-minute SMJ on 100GB is expected; a 30-minute BroadcastHashJoin on 10MB is a bug.
- Use Spark History Server's `EventLog` JSON to build automated alerting: parse stage metrics and alert on spill > 0, GC > 10%, or max/median ratio > 5.
- For streaming jobs, the Spark UI Streaming tab shows batch latency, processing time, and scheduling delay — scheduling delay > 0 means the cluster cannot keep up.
- The `Thread Dump` button on the Executors tab is invaluable for debugging stuck tasks — look for threads blocked on shuffle read or GC.

In [ ]:
# Production UI metrics collection via Spark internal API
def collect_job_metrics(spark):
    sc = spark.sparkContext
    status = sc.statusTracker()

    job_ids = status.getJobIdsForGroup()
    print(f'=== Job Metrics (statusTracker) ===')
    print(f'  Active jobs   : {len(status.getActiveJobsIds())}')
    print(f'  Active stages : {len(status.getActiveStageIds())}')

    executor_infos = status.getExecutorInfos()
    print(f'  Active executors: {len(executor_infos)}')
    for ei in executor_infos:
        print(f'    Executor {ei.executorId()}: host={ei.host()}')

    # Print key configs that affect UI-visible metrics
    print('\n  Key configs affecting UI metrics:')
    ui_configs = [
        'spark.sql.adaptive.enabled',
        'spark.sql.shuffle.partitions',
        'spark.sql.autoBroadcastJoinThreshold',
        'spark.eventLog.enabled',
    ]
    for c in ui_configs:
        print(f'    {c} = {spark.conf.get(c, "default")}')

collect_job_metrics(spark)

### Practice Exercises — Spark UI

1. Create an intentionally skewed dataset: 90% of rows have `key=1`, 10% spread across 99 other keys. Run a groupBy aggregation and observe the stage task duration chart. Enable AQE skew join handling and re-run. Document before/after Max/Median task time ratio from the Stages tab.
2. Introduce a deliberate disk spill: create a dataset that requires more sort memory than available by using a very small `spark.driver.memory` or by sorting 2M rows with many columns. Confirm `Spill (Disk) > 0` in the Stages tab. Then increase `spark.sql.shuffle.partitions` to reduce per-partition sort size and confirm spill disappears.
3. Use the History Server REST API (`/api/v1/applications/{appId}/stages`) to programmatically extract stage metrics for 3 of your test jobs. Build a function `def parse_stage_metrics(json_response)` that returns a dict with: max_task_time, median_task_time, total_spill_disk, total_shuffle_read, skew_ratio.

In [ ]:
# Clean up
spark.stop()
print('SparkSession stopped.')